In [ ]:
# Imports
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pyarrow"], check=True)

import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
import numpy as np

### 1) Directory Structure

In [2]:
load_dotenv()

api_key = os.getenv("API_KEY")
repo_owner = 'bedbad'
repo_name = 'justpyplot'

url = "https://api.github.com/graphql"
headers = {
    "Authorization": f"Bearer {api_key}"
}

# Rest of code in this cell is partly AI generated
# "... on Tree" is an Inline fragment. If the object is a Tree, it will fetch the entries.
def get_folder_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Tree {
            entries {
              name
              path
              type
              extension
            }
          }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

all_items = []
# Start at the root of the repository
folders_to_check = ["HEAD:"]

while folders_to_check:
    current_folder = folders_to_check.pop(0)
    data = get_folder_contents(current_folder)
    try:
        entries = data['data']['repository']['object']['entries']
    except (KeyError, TypeError):
        continue
        
    for entry in entries:
        all_items.append({
            'Name': entry['name'],
            'Path': entry['path'],
            'Type': entry['type'], # 'blob' for files, 'tree' for folders
            'Extension': entry.get('extension', '')
        })
        
        # If the item is a folder (tree), queue it up to fetch its contents next
        if entry['type'] == 'tree':
            folders_to_check.append(f"HEAD:{entry['path']}/")

# Dump a sample to json for reference
with open("directory_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total files and folders.")

Found 38 total files and folders.


In [3]:
df = pd.DataFrame(all_items)

# Change type column to something human readible
df['Type'] = np.where(df['Type'] == 'tree', 'folder', 'file')

# Sort alphabetically by path
df_folders = df.sort_values(by='Path').reset_index(drop=True)

print("Created CSV File")
df_folders.head(5)

Created CSV File


,Name,Path,Type,Extension
0,.github,.github,folder,
1,funding.yaml,.github/funding.yaml,file,.yaml
2,workflows,.github/workflows,folder,
3,python-publish.yml,.github/workflows/python-publish.yml,file,.yml
4,.gitignore,.gitignore,file,


### 2) Naming Conventions

In [4]:
def get_file_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Blob {
              text
            }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

# We're only extracting Python repos. So get all .py files
df_files = df.loc[df['Extension'] == '.py', ['Path']]

# Prefix with HEAD: for the query
df_files['Path'] = 'HEAD:' + df_files['Path']
files_to_check = df_files.squeeze().tolist()

print(files_to_check)
all_items = []

while files_to_check:
    current_file = files_to_check.pop()
    data = get_file_contents(current_file)
    try:
        raw_text = data['data']['repository']['object']['text']
    except (KeyError, TypeError):
        continue
    
    # Remove the prefix HEAD: from the query
    current_file = current_file[5:]
    all_items.append({
        'Path': current_file,
        'Raw text': raw_text
    })

# Dump a sample to json for reference
with open("file_query.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Found {len(all_items)} total Python files.")

['HEAD:docs/conf.py', 'HEAD:examples/__init__.py', 'HEAD:justpyplot/__init__.py', 'HEAD:justpyplot/justpyplot.py', 'HEAD:justpyplot/textrender.py', 'HEAD:tests/__init__.py', 'HEAD:tests/test.py', 'HEAD:tests/test_basic.py', 'HEAD:tests/test_basic_plot.py', 'HEAD:tests/test_plot.py', 'HEAD:tests/test_plot_components.py', 'HEAD:tests/test_standalone.py', 'HEAD:tests/test_textrender.py', 'HEAD:examples/mug_objectron/__init__.py', 'HEAD:examples/mug_objectron/demo.py']
Found 15 total Python files.


In [5]:
# Migrate to Pandas Dataframe
df_files = pd.DataFrame(all_items)

# Merge with the original folders dataframe from Section 1.
df_folder_and_files = pd.merge(df_folders, df_files, on='Path')

# Final dataframe for Section 2.
df_folder_and_files.head(5)

# CSV files don't really work with rawtext bc of commas so save as a parquet instead
# IMPORTANT: Make sure to run pip install pyarrow
df_folder_and_files.to_parquet('dataset.parquet')

### 3) Test Suite Metrics

In [ ]:
import ast

def analyze_test_metrics(path, raw_text):
    """
    Parses a Python file and extracts per-file test suite metrics using the AST.
    
    A file is classified as a test file if its name starts with 'test_',
    ends with '_test.py', or lives inside a directory named 'test' or 'tests'.
    """
    filename = path.split('/')[-1]
    path_parts = set(path.split('/'))

    is_test_file = (
        filename.startswith('test_') or
        filename.endswith('_test.py') or
        bool({'test', 'tests'} & path_parts)
    )

    metrics = {
        'is_test_file': is_test_file,
        'line_count': len(raw_text.splitlines()),
        'test_function_count': 0,   # functions named test_*
        'source_function_count': 0, # all other functions
        'assertion_count': 0,       # ast.Assert nodes
        'uses_pytest': False,
        'uses_unittest': False,
        'parse_error': False,
    }

    try:
        tree = ast.parse(raw_text)
    except SyntaxError:
        metrics['parse_error'] = True
        return metrics

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            if node.name.startswith('test_'):
                metrics['test_function_count'] += 1
            else:
                metrics['source_function_count'] += 1

        elif isinstance(node, ast.Assert):
            metrics['assertion_count'] += 1

        elif isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name == 'pytest':
                    metrics['uses_pytest'] = True
                elif alias.name == 'unittest':
                    metrics['uses_unittest'] = True

        elif isinstance(node, ast.ImportFrom):
            if node.module and 'pytest' in node.module:
                metrics['uses_pytest'] = True
            elif node.module and 'unittest' in node.module:
                metrics['uses_unittest'] = True

    return metrics

# Load the dataset saved in Section 2 and apply per-file analysis
df = pd.read_parquet('dataset.parquet')

file_metrics = df.apply(
    lambda row: analyze_test_metrics(row['Path'], row['Raw text']),
    axis=1,
    result_type='expand'
)

df_metrics = pd.concat([df[['Path', 'Name', 'Extension']], file_metrics], axis=1)
df_metrics

In [ ]:
test_files   = df_metrics[df_metrics['is_test_file']]
source_files = df_metrics[~df_metrics['is_test_file']]

total_py_files        = len(df_metrics)
total_test_functions  = int(df_metrics['test_function_count'].sum())
total_source_functions = int(df_metrics['source_function_count'].sum())
total_assertions      = int(df_metrics['assertion_count'].sum())
total_test_lines      = int(test_files['line_count'].sum())

repo_metrics = {
    # File-level counts
    'total_py_files':           total_py_files,
    'test_file_count':          len(test_files),
    'source_file_count':        len(source_files),

    # Ratios
    'test_file_ratio':          len(test_files) / total_py_files if total_py_files > 0 else 0,
    'test_to_source_fn_ratio':  total_test_functions / total_source_functions if total_source_functions > 0 else 0,

    # Raw counts
    'total_test_functions':     total_test_functions,
    'total_source_functions':   total_source_functions,
    'total_assertions':         total_assertions,

    # Assertions per line of test code (proxy for how thoroughly tests check behaviour)
    'assertion_density':        total_assertions / total_test_lines if total_test_lines > 0 else 0,

    # Framework detection
    'uses_pytest':              bool(df_metrics['uses_pytest'].any()),
    'uses_unittest':            bool(df_metrics['uses_unittest'].any()),
}

print("=== Repo-Level Test Suite Metrics ===")
for k, v in repo_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Save per-file metrics alongside existing dataset columns
df_metrics.to_parquet('dataset_with_test_metrics.parquet')

# Save the repo-level summary separately for easy downstream use
with open('repo_test_metrics.json', 'w') as f:
    json.dump(repo_metrics, f, indent=4)

print("\nSaved dataset_with_test_metrics.parquet and repo_test_metrics.json")